# Geoloc



## infos
Participants : Hélène DALON-DENEE, Dame DIENG, Célien GRIL, Nathan LEBRE

ligne 42368 : photo id 5464485473, correction -> les dates étaient décalée, le # de minutes (25) était collé au titre de la photo ("une lundi matin comme tout les autre ;-(") et le décalage était propagé.

In [1]:
import numpy as np
import pandas as pd
import folium as fl
import hdbscan
import re
import matplotlib.pyplot as plt

In [2]:
read_data = pd.read_csv("../flickr_data2.csv")
len(read_data)

C:\Users\Pyta\AppData\Local\Temp\ipykernel_20800\1384916255.py:1: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  read_data = pd.read_csv("../flickr_data2.csv")


420240

Retirer les 142 lignes qui ont des valeurs non-nulles qui dépassent les colonnes attendues. (unnamed 16, y'a aussi 2 lignes avec des valeurs dans unnamed 18 mais elles sont comptabilisées dans les autres)

In [3]:
wrong_data = read_data.dropna(how="all", subset=read_data.columns[[16, 18]])
fix_data = read_data.drop(index=wrong_data.index, axis=1)

toDropColumns = read_data.columns[[16, 17, 18]]
fix_data = fix_data.drop(toDropColumns, axis=1)

len(fix_data)

420098

Retirer les tuples dupliquée (ignore id et tags pour enlever certains dupliqué quand un post est modifié et certains carousels, passe de 187544 à 175688 donc pas tant que ça, la majorité sont dupliqués point barre)

In [4]:
fix_data.drop_duplicates(inplace=True, subset=fix_data.columns.drop(["id", " tags"]), keep='last')
len(fix_data)

175689

Retirer les carousels

In [5]:
fix_data.drop_duplicates(inplace=True, subset=fix_data.columns.drop(["id", " title"]))
len(fix_data)

161123

Retirer les tags null pour le text pattern mining

In [6]:
text_data = fix_data.dropna(how='all', subset=[" tags"])
len(text_data)

120617

## Clustering

### HBDSCAN

In [7]:
data2D = fix_data[[' lat', ' long']]
coords_rad = np.radians(data2D)
coords_rad.head()

,lat,long
9420,0.798799,0.084408
9421,0.798799,0.084408
9422,0.798674,0.084585
9423,0.798677,0.084250
9424,0.798696,0.084237


In [8]:
clusterer = hdbscan.HDBSCAN(min_cluster_size = 100, min_samples = 50, metric= 'haversine')
labels = clusterer.fit_predict(coords_rad)
df = data2D.copy()
df["cluster"] = labels
pois = (
    df[df.cluster != -1]
    .groupby("cluster")
    .agg(
        lat_mean=(" lat", "mean"),
        lon_mean=(" long", "mean"),
        nb_photos=("cluster", "count")
    )
    .sort_values("nb_photos", ascending=False)
)

## Partie sur la carte.

In [9]:
# Réinitialiser les indices pour éviter les problèmes
donnees_geoloc_reset = data2D.reset_index(drop=True)
labels_reset = labels.copy()

# Créer la carte centrée sur Lyon
map_hdbscan = fl.Map(
    location=[45.757778, 4.832222],
    zoom_start=12
)

# Couleurs pour les clusters
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 
          'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'gray']

# Ajouter les points avec les couleurs des clusters
for idx in range(len(donnees_geoloc_reset)):
    cluster_id = labels_reset[idx]
    row = donnees_geoloc_reset.iloc[idx]
    
    # Sauter le bruit (cluster -1)
    if cluster_id == -1:
        continue
    
    # Choisir la couleur selon le cluster
    color = colors[int(cluster_id) % len(colors)]
    
    # Ajouter un marqueur
    fl.CircleMarker(
        location=[row[' lat'], row[' long']],
        radius=3,
        popup=f'Cluster {cluster_id}',
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.7,
        weight=1
    ).add_to(map_hdbscan)

# Ajouter les points de bruit en gris transparent
for idx in range(len(donnees_geoloc_reset)):
    if labels_reset[idx] == -1:
        row = donnees_geoloc_reset.iloc[idx]
        fl.CircleMarker(
            location=[row[' lat'], row[' long']],
            radius=1,
            popup='Bruit',
            color='gray',
            fill=True,
            fillColor='gray',
            fillOpacity=0.2,
            weight=0.5
        ).add_to(map_hdbscan)

# Ajouter les centres des clusters (POIs)
for cluster_id, poi_row in pois.iterrows():
    fl.Marker(
        location=[poi_row['lat_mean'], poi_row['lon_mean']],
        popup=f"Cluster {cluster_id}<br>{int(poi_row['nb_photos'])} photos",
        icon=fl.Icon(color=colors[int(cluster_id) % len(colors)], icon='info-sign')
    ).add_to(map_hdbscan)

# Sauvegarder la carte
map_hdbscan.save('map_hdbscan.html')
print("Carte HDBSCAN sauvegardée dans 'map_hdbscan.html'")
print(f"\nRésumé des clusters HDBSCAN:")
print(pois)

Carte HDBSCAN sauvegardée dans 'map_hdbscan.html'

Résumé des clusters HDBSCAN:
          lat_mean  lon_mean  nb_photos
cluster                                
32       45.837448  4.826248       3748
90       45.732627  4.818266       2353
95       45.741761  4.817054       2174
33       45.837410  4.826163       2159
297      45.760934  4.826798       2025
...            ...       ...        ...
100      45.780521  4.872484        103
7        45.690163  4.786463        102
107      45.769953  4.862777        102
274      45.766609  4.826910        101
114      45.771234  4.845757        100

[299 rows x 3 columns]


# Text mining

In [10]:
nona_df = text_data[' tags'].dropna()

#count = nona_df.apply(lambda x: len(re.findall(r'[^\W\d_]++', str(x)))).sum()
#print(f"Number of tags in the cleaned dataset: {count}")
print(nona_df)

9420                                        lyon,operahouse
9421                                        lyon,operahouse
9422                                  365,rubberduck,iphone
9423                                         lyon,vieuxlyon
9424                                         lyon,vieuxlyon
                                ...                        
420234    europe,france,lyon,croixrousse,streetart,wall,...
420235    europe,france,lyon,croixrousse,streetart,wheat...
420237    auvergnerhônealpes,rhône,lyonnais,valléedurhôn...
420238    auvergnerhônealpes,rhône,lyonnais,valléedurhôn...
420239    ngc,lyon,paysage,landscape,ville,urbain,town,tour
Name:  tags, Length: 120617, dtype: object


In [11]:
idf={}  # Inverse Document Frequency dictionary
total_docs = len(nona_df)
filtered_tags=["cospla","japa","feminicide","girl","hair","overwatch","tracer","aplusphoto","view","fiume","night","chaise","chair","iphone","streetphotography","rue","gens","nuit","francia","poste","river","notte","night"]

for tags in nona_df:
    unique_tags = set(re.findall(r'[^\W\d_]{2,}', str(tags)))    
    unique_tags = {tag.lower() for tag in unique_tags if not any(bad in tag for bad in filtered_tags)}  # Normalize to lowercase
    for tag in unique_tags:
        idf[tag] = idf.get(tag, 0) + 1

for tag in idf:
    idf[tag] = total_docs / idf[tag]

print("IDF values for tags:")
for tag, value in idf.items():
    print(f"{tag}: {value}")

IDF values for tags:
operahouse: 2566.31914893617
lyon: 1.5737920956146187
rubberduck: 60308.5
vieuxlyon: 49.514367816091955
parcdeshauteurs: 7095.117647058823
vert: 299.29776674937966
heart: 1507.7125
green: 249.7246376811594
coeur: 1698.8309859154929
flore: 4020.5666666666666
cortex: 4159.206896551724
fleur: 223.7792207792208
red: 217.72021660649818
rouge: 204.78268251273346
macro: 210.50087260034903
closeup: 2412.34
flower: 225.45233644859812
georges: 1453.2168674698796
bistrot: 2741.2954545454545
restaurant: 365.5060606060606
bouchon: 1608.2266666666667
resto: 1945.4354838709678
saint: 272.8891402714932
brasserie: 2319.5576923076924
choucroute: 120617.0
christianbaudet: 1127.2616822429907
fourvière: 45.74023511566174
geotagged: 103.71195184866724
parctêtedor: 2116.0877192982457
panorama: 115.53352490421456
bokeh: 391.6136363636364
day: 353.7155425219941
candle: 3890.8709677419356
baramericain: 60308.5
square: 12.652575264869402
turquoise: 8615.5
goldeneyes: 120617.0
january: 6030.8

In [12]:
#Cleaning by the lower IDF values

threshold = 18  # Default threshold
filtered_tags = []

idf = {tag: value for tag, value in idf.items() if value >= threshold}
count= len(idf)
print(f"Number of tags after filtering with threshold {threshold}: {count}")
print("IDF values for tags:")
for tag, value in idf.items():
    print(f"{tag}: {value}")

tag_df=pd.DataFrame(list(idf.items()), columns=['tag', 'idf_value'])

Number of tags after filtering with threshold 18: 39795
IDF values for tags:
operahouse: 2566.31914893617
rubberduck: 60308.5
vieuxlyon: 49.514367816091955
parcdeshauteurs: 7095.117647058823
vert: 299.29776674937966
heart: 1507.7125
green: 249.7246376811594
coeur: 1698.8309859154929
flore: 4020.5666666666666
cortex: 4159.206896551724
fleur: 223.7792207792208
red: 217.72021660649818
rouge: 204.78268251273346
macro: 210.50087260034903
closeup: 2412.34
flower: 225.45233644859812
georges: 1453.2168674698796
bistrot: 2741.2954545454545
restaurant: 365.5060606060606
bouchon: 1608.2266666666667
resto: 1945.4354838709678
saint: 272.8891402714932
brasserie: 2319.5576923076924
choucroute: 120617.0
christianbaudet: 1127.2616822429907
fourvière: 45.74023511566174
geotagged: 103.71195184866724
parctêtedor: 2116.0877192982457
panorama: 115.53352490421456
bokeh: 391.6136363636364
day: 353.7155425219941
candle: 3890.8709677419356
baramericain: 60308.5
turquoise: 8615.5
goldeneyes: 120617.0
january: 60

In [13]:
mining_data = df.copy()
tags_data = tag_df["tag"].str.split()
mining_data = mining_data.merge(tags_data, left_index=True, right_index=True)
mining_data = mining_data.dropna(subset=["tag"])


In [14]:
print(mining_data)

             lat      long  cluster                         tag
9420   45.767811  4.836215      257             [michelinguide]
9421   45.767811  4.836215      257                [gastronomy]
9422   45.760649  4.846391      122                 [bocusedor]
9423   45.760846  4.827150      297           [papedelacuisine]
9424   45.761924  4.826399       -1  [aubergedupontdecollonges]
...          ...       ...      ...                         ...
39787  45.837448  4.826248       32                    [spathe]
39788  45.837448  4.826248       32                 [anthurium]
39789  45.837448  4.826248       32              [grandsserres]
39792  45.837448  4.826248       32                     [dekor]
39793  45.837448  4.826248       32                     [wagen]

[10689 rows x 4 columns]


In [15]:
#erreur pour le moment
tags_dict = {}
for row in mining_data.itertuples():
    clust = getattr(row, 'cluster')
    tags = getattr(row, 'tag')
    if tags_dict.keys().__contains__(clust):
        tags_clust = tags_dict[clust]
        for tag in tags:
            #print(tag)
            if tags_clust.keys().__contains__(tag):
                tags_clust[tag] += 1
            else:
                tags_clust[tag] = 1
        tags_dict[clust] = tags_clust
    else:
        tags_clust = {}
        for tag in tags:
            if tags_clust.keys().__contains__(tag):
                tags_clust[tag] += 1
            else:
                tags_clust[tag] = 1
        tags_dict[clust] = tags_clust


In [16]:
tags_dict

{257: {'michelinguide': 1,
  'gastronomy': 1,
  'debrousse': 1,
  'loyon': 1,
  'lyonlightsfestival': 1,
  'odeaubois': 1,
  'bicicleta': 1,
  'ludtzcanon': 1,
  'fantom': 1,
  'mp': 1,
  'ferdinand': 1,
  'अर': 1,
  'カオスのすみか': 1,
  'toiletpaper': 1,
  'ddcee': 1,
  'lamouche': 1,
  'menestrier': 1,
  'bau': 1,
  'bô': 1,
  'passerelledupalaisdejusticelyon': 1,
  'michaelelmgreen': 1,
  'moulinàvent': 1,
  'mmsummicronv': 1,
  'hôtelroyal': 1,
  'brunante': 1,
  'beg': 1,
  'businova': 1,
  'quaiperrache': 1,
  'clio': 1,
  'therepublicanspoliticalparty': 1,
  'therepublicans': 1,
  'investmentbanker': 1,
  'banquier': 1,
  'lesrougonmacquart': 1,
  'highheels': 1,
  'lysistrata': 1,
  'mauves': 1,
  'backside': 1,
  'neonsign': 1,
  'vx': 1,
  'coursieravelo': 1,
  'payant': 1,
  'obscurworld': 1,
  'audessus': 1,
  'envitrine': 1,
  'sprocket': 1,
  'abstact': 1,
  'citadine': 1,
  'picchiorossomaggiore': 1,
  'limonade': 1,
  'visages': 1,
  'adagioappassionato': 1,
  'romà': 1,
  '

In [17]:
# Réinitialiser les indices pour éviter les problèmes
donnees_geoloc_reset = data2D.reset_index(drop=True)
labels_reset = labels.copy()

# Créer la carte centrée sur Lyon
map_hdbscan = fl.Map(
    location=[45.757778, 4.832222],
    zoom_start=12
)

# Couleurs pour les clusters
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 
          'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'gray']

# Ajouter les points avec les couleurs des clusters
for idx in range(len(donnees_geoloc_reset)):
    cluster_id = labels_reset[idx]
    row = donnees_geoloc_reset.iloc[idx]
    
    # Sauter le bruit (cluster -1)
    if cluster_id == -1:
        continue
    
    # Choisir la couleur selon le cluster
    color = colors[int(cluster_id) % len(colors)]
    
    # Ajouter un marqueur
    fl.CircleMarker(
        location=[row[' lat'], row[' long']],
        radius=3,
        popup=f'Cluster {cluster_id}',
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.7,
        weight=1
    ).add_to(map_hdbscan)

# Ajouter les points de bruit en gris transparent
for idx in range(len(donnees_geoloc_reset)):
    if labels_reset[idx] == -1:
        row = donnees_geoloc_reset.iloc[idx]
        fl.CircleMarker(
            location=[row[' lat'], row[' long']],
            radius=1,
            popup='Bruit',
            color='gray',
            fill=True,
            fillColor='gray',
            fillOpacity=0.2,
            weight=0.5
        ).add_to(map_hdbscan)

# Ajouter les centres des clusters (POIs)
for cluster_id, poi_row in pois.iterrows():
    fl.Marker(
        location=[poi_row['lat_mean'], poi_row['lon_mean']],
        popup=f"Cluster {cluster_id}<br>{int(poi_row['nb_photos'])} photos <br>Tags: {', '.join(tags_dict.get(cluster_id, {}).keys())}",
        icon=fl.Icon(color=colors[int(cluster_id) % len(colors)], icon='info-sign')
    ).add_to(map_hdbscan)

# Sauvegarder la carte
map_hdbscan.save('map_tag_on.html')
print("Carte HDBSCAN sauvegardée dans 'map_tag_on.html'")
print(f"\nRésumé des clusters HDBSCAN:")
print(pois)

Carte HDBSCAN sauvegardée dans 'map_tag_on.html'

Résumé des clusters HDBSCAN:
          lat_mean  lon_mean  nb_photos
cluster                                
32       45.837448  4.826248       3748
90       45.732627  4.818266       2353
95       45.741761  4.817054       2174
33       45.837410  4.826163       2159
297      45.760934  4.826798       2025
...            ...       ...        ...
100      45.780521  4.872484        103
7        45.690163  4.786463        102
107      45.769953  4.862777        102
274      45.766609  4.826910        101
114      45.771234  4.845757        100

[299 rows x 3 columns]


## Scope of time

Étudions les évenemments lyonnais

### 1.Préparation des données


In [18]:
data_time = fix_data[[' date_taken_day', ' date_taken_month', ' lat', ' long']]
print(data_time)

        date_taken_day  date_taken_month        lat      long
9420                31                 8  45.767811  4.836215
9421                31                 8  45.767811  4.836215
9422                21                 2  45.760649  4.846391
9423                10                 5  45.760846  4.827150
9424                10                 5  45.761924  4.826399
...                ...               ...        ...       ...
420235              30                 9  45.758316  4.825197
420236               5                10  45.762635  4.837299
420237              27                 9  45.763657  4.836012
420238              27                 9  45.763657  4.836012
420239              28                 9  45.758181  4.831967

[161123 rows x 4 columns]


In [19]:
count_month= data_time[' date_taken_month'].value_counts().sort_index()
df_month = {mois : i for mois, i in data_time.groupby(' date_taken_month')}
print(df_month)

{1:         date_taken_day  date_taken_month        lat      long
9438                31                 1  45.781748  4.855077
9439                31                 1  45.781748  4.855077
9440                31                 1  45.782975  4.854476
9441                31                 1  45.780259  4.849820
9445                22                 1  45.768211  4.854766
...                ...               ...        ...       ...
417924              20                 1  45.757797  4.831586
417925              20                 1  45.756638  4.833027
417926              20                 1  45.756925  4.832694
417927              20                 1  45.756658  4.832022
417928              20                 1  45.756647  4.831366

[9238 rows x 4 columns], 2:         date_taken_day  date_taken_month        lat      long
9422                21                 2  45.760649  4.846391
9433                21                 2  45.773569  4.851322
9434                21               

In [ ]:
# Réinitialiser les indices pour éviter les problèmes
donnees_geoloc_reset = data2D.reset_index(drop=True)
labels_reset = labels.copy()

# Créer la carte centrée sur Lyon
map_hdbscan = fl.Map(
    location=[45.757778, 4.832222],
    zoom_start=12
)

# Couleurs pour les clusters
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 
          'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'gray']

# Ajouter les points avec les couleurs des clusters
for idx in range(len(donnees_geoloc_reset)):
    cluster_id = labels_reset[idx]
    row = donnees_geoloc_reset.iloc[idx]
    
    # Sauter le bruit (cluster -1)
    if cluster_id == -1:
        continue
    
    # Choisir la couleur selon le cluster
    color = colors[int(cluster_id) % len(colors)]
    
    # Ajouter un marqueur
    fl.CircleMarker(
        location=[row[' lat'], row[' long']],
        radius=3,
        popup=f'Cluster {cluster_id}',
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.7,
        weight=1
    ).add_to(map_hdbscan)

# Ajouter les points de bruit en gris transparent
for idx in range(len(donnees_geoloc_reset)):
    if labels_reset[idx] == -1:
        row = donnees_geoloc_reset.iloc[idx]
        fl.CircleMarker(
            location=[row[' lat'], row[' long']],
            radius=1,
            popup='Bruit',
            color='gray',
            fill=True,
            fillColor='gray',
            fillOpacity=0.2,
            weight=0.5
        ).add_to(map_hdbscan)

# Ajouter les centres des clusters (POIs)
for cluster_id, poi_row in pois.iterrows():
    fl.Marker(
        location=[poi_row['lat_mean'], poi_row['lon_mean']],
        popup=f"Cluster {cluster_id}<br>{int(poi_row['nb_photos'])} photos",
        icon=fl.Icon(color=colors[int(cluster_id) % len(colors)], icon='info-sign')
    ).add_to(map_hdbscan)

# Sauvegarder la carte
map_hdbscan.save('map_hdbscan.html')
print("Carte HDBSCAN sauvegardée dans 'map_hdbscan.html'")
print(f"\nRésumé des clusters HDBSCAN:")
print(pois)